<a href="https://colab.research.google.com/github/mehrdadriahi/jet/blob/main/AIVOICE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ## 🚀 So-VITS-SVC Colab Setup & Training Notebook

---

### 📌 Cell 1: بررسی و مطمئن شدن از GPU
```python
# مطمئن شدن که GPU در دسترس است
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
# کلون کردن آخرین نسخه و رفتن به فولدر پروژه
!git clone https://github.com/svc-develop-team/so-vits-svc.git
%cd so-vits-svc


In [ ]:
# بروز کردن pip و نصب پکیج‌های پایه
!python -m pip install --upgrade pip setuptools numpy numba
!pip install pyworld==0.3.1 praat-parselmouth
!pip install librosa==0.8.1 soundfile tqdm webrtcvad


In [ ]:
# اگر قبلاً fairseq نصب بود، پاکش کن
!pip uninstall -y fairseq

# نصب نسخه پچ‌شده (liyaodev) به همراه هیدرا و اومگاکنف
!pip install git+https://github.com/liyaodev/fairseq.git
!pip install hydra-core==1.3.2 omegaconf==2.3.0


In [ ]:
%%bash
PY_FILE=$(python -c "import os; import fairseq; print(os.path.dirname(fairseq.__file__)+'.dist-info')"); exit
# اضافه کردن whitelist برای Dictionary و weights_only=False
sed -i '1iimport torch\nfrom fairseq.data.dictionary import Dictionary\ntorch.serialization.add_safe_globals([Dictionary])' \
    /usr/local/lib/python3.11/dist-packages/fairseq/checkpoint_utils.py
sed -i \
  's/torch.load(f, map_location=torch.device("cpu"))/torch.load(f, map_location=torch.device("cpu"), weights_only=False)/' \
  /usr/local/lib/python3.11/dist-packages/fairseq/checkpoint_utils.py


In [ ]:
# پس از ری‌استارت شدن
%cd /content/so-vits-svc
!pip install torchcrepe tensorboardX


In [ ]:
# پوشه ی pretrain بساز
!mkdir -p pretrain

# دانلود ContentVec (checkpoint_best_legacy_500.pt)
!wget -q -P pretrain \
  https://huggingface.co/therealvul/so-vits-svc-4.0-init/resolve/main/checkpoint_best_legacy_500.pt \
  -O pretrain/checkpoint_best_legacy_500.pt

# دانلود RMVPE و تغییر نام
!wget -q -P pretrain \
  https://github.com/yxlllc/RMVPE/releases/download/230917/rmvpe.zip
!unzip -o pretrain/rmvpe.zip -d pretrain
!mv pretrain/model.pt pretrain/rmvpe.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# اگر بار اول است و فایل raw zip را دارید:
!mkdir -p /content/so-vits-svc/dataset_raw
!unzip -o "/content/drive/MyDrive/hamid.zip" -d /content/so-vits-svc/dataset_raw

# ————————
# یا اگر قبلاً پری‌پروسس کردید و بک‌آپ دارید:
# back_up_name = "hamid_traiined"
# BACK_UP="/content/drive/MyDrive/dataset/$back_up_name"
# !unzip -o $BACK_UP/dataset.zip -d /
# !cp $BACK_UP/config.json configs/config.json
# !cp $BACK_UP/train.txt filelists/train.txt
# !cp $BACK_UP/val.txt   filelists/val.txt


In [ ]:
# اگر نیاز به resample دارید
!python resample.py


In [ ]:
!pip install loguru
!python preprocess_flist_config.py


In [ ]:
!python preprocess_hubert_f0.py


In [ ]:
# فشرده‌سازی دیتاست نهایی
!zip -r dataset.zip /content/so-vits-svc/dataset

# نام دلخواه برای پوشه روی Drive:
dataset_name_drive = "hamid_traiined"

# مسیر مقصد
DATA_DST="/content/drive/MyDrive/dataset/$dataset_name_drive"
mkdir -p "$DATA_DST"

# کپی فایل‌ها
cp dataset.zip    "$DATA_DST/"
cp configs/config.json      "$DATA_DST/"
cp filelists/train.txt      "$DATA_DST/"
cp filelists/val.txt        "$DATA_DST/"


In [ ]:
# نام کلون/رزولوشن (مثلاً 44k)
Clone="44k"

# لینک فولدر لاگ‌ها به Drive
mkdir -p "/content/drive/MyDrive/$Clone"
rm -rf "logs/$Clone"
ln -s "/content/drive/MyDrive/$Clone" "logs/$Clone"


In [ ]:
# اگر اولین راند آموزش است:
wget -q -P "logs/$Clone" \
  https://huggingface.co/therealvul/so-vits-svc-4.0-init/resolve/main/G_0.pth \
  -O "logs/$Clone/G_0.pth"
wget -q -P "logs/$Clone" \
  https://huggingface.co/therealvul/so-vits-svc-4.0-init/resolve/main/D_0.pth \
  -O "logs/$Clone/D_0.pth"


In [ ]:
# فعال کردن TensorBoard (اختیاری)
%load_ext tensorboard
%tensorboard --logdir logs/$Clone

# کامند آموزش
python train.py -c configs/config.json -m "$Clone"


In [ ]:
python cluster/train_cluster.py


In [ ]:
# به‌عنوان مثال G و D پس از N گام
cp logs/$Clone/G_1000.pth "/content/drive/MyDrive/$Clone/G_1000.pth"
cp logs/$Clone/D_1000.pth "/content/drive/MyDrive/$Clone/D_1000.pth"
